# P-2026-11-23-1: How fragile are the report countries' HS4 ECI ranks to products sitting near the RCA = 1 threshold?

**Question** (from `proposals/2026-11-22.md` #1, open PR branch `claude/nightly-2026-11-22`): for each Spring 2026 report country and its report peers, how many RCA ≥ 1 products sit just above the threshold (RCA in [1, 1.15)) or just below it (RCA in [1/1.15, 1)), and how far does the oriented HS4 ECI rank move when those products flip, **one at a time** and **all together**? Do the reports' peer orderings and 2019 → 2022 direction calls survive?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`, so per PROTOCOL §1.4 the question comes from `proposals/`. Run folders on `main` and on the open branches `claude/nightly-2026-11-20/21/22` already claim P-11-13 … P-11-22. 11-21 #4 asks Will to close the resource-inside-ECI line, so tonight I skipped 11-19 #1/#2 and 11-20 #2. 11-22 #1 is the newest proposal that scores neither an ECI variant nor a `lib/` change, and it checks something we already observed: Vietnam's integrated circuits (8542) sit at RCA 1.106.

- **Date:** 2026-11-23 (America/New_York), nightly run. ID `P-2026-11-23-1`.
- **Data:** `data/raw/2019–2023_mega.csv` → HS4 via `lib/research.repo_hs` (chapters 98/99 dropped, reporters ≥ US$1B, ISO3). No external data. No downloads.
- **Years:** 2019 primary (all 133 countries); 2020–2022 checks on the 20 focus countries; 2023 is loaded (Section A only) but not used for any conclusion (D1).
- **Method settings:** the same as 11-20 to 11-22: RCA ≥ 1, sorted SVD, O4 orientation (PCI negative with ubiquity). `lib/eci.py` and `lib/research.py` are imported but not modified.
- **Peers:** the dict from the 11-14/11-15 runs (taken from the reports' peer sentences).
- **Run time:** printed in the last code cell (about 4.5 minutes, mostly the 4,706 exact SVD recomputes).

## TL;DR

- **No single near-threshold product moves a report country's HS4 rank by more than 2 places.** In 2019, flipping any one band product (RCA within 15% of 1) moves the report countries by at most **0–2 ranks** (UAE 2; Germany and Singapore 0) and **0.009–0.051 SD** of ECI. Over 2019–2022 the maximum is still 0–2. Vietnam's integrated circuits (8542, RCA 1.106, PCI 1.776, US$11.5B) cost **1 rank (−0.021 SD)** when flipped out. All 7 report countries' largest single flip is smaller than the median one-year ECI move (**0.084 SD**, 2019→20).
- **Own-threshold fragility is concentrated in small-basket commodity exporters, not in the report countries.** Across 131 countries with band products, the largest single-flip |ΔECI| falls with diversity (Spearman **−0.896**). 20 of 131 countries have a flip that moves ≥ 3 ranks, and 6 have one that moves ≥ 5 (Congo 15, Angola 10, Kuwait 7, Qatar 6, Zambia 5, Gabon 5). Two of those are UAE peers.
- **Moving the whole band at once matters more for some countries.** The joint scenario raises or lowers one country's RCA threshold to 1.15 or 1/1.15 while everyone else stays fixed. Rank intervals in 2019: Canada **40–49**, Vietnam **58–63**, UK **10–14**, Poland 30–32, UAE 56–57, Germany 6, Singapore 5. Canada is the most fragile report country: it has 45 near-misses whose mean PCI (0.256) is well above its basket's mean (−0.203). Tightening the threshold *raises* ECI for 7 of 20 focus countries, so "fragile" doesn't mean "overstated".
- **The reports' peer orderings are robust and their direction calls mostly are not.** **20 of 21** report-vs-peer orderings survive the band in 2019 (and in 2020 and 2021; 17/20 in 2022). The exceptions are Singapore vs Korea (2019, 2020, 2022), UAE vs Kuwait (2021, 2022) and UAE vs Qatar (2022). Only **2 of 7** 2019 → 2022 direction calls (Poland and Singapore, both down) have disjoint intervals. This agrees with 11-14, which found many direction calls are noise.
- **Last night's 22-rank Vietnam fall is mostly a system-wide effect.** Applying the H1 re-basing factor (f = 0.891, 20 products leave) to **Vietnam's row only** costs it 5 ranks (58 → 63, −0.067 SD). For the lab: HS4 ranks are safe from a country's *own* threshold noise. Common shocks that move everyone's RCA are the risk (11-16, 11-22).

## Why this matters for the lab

This tests agenda §0 link 3 (the optimizer's target is sound) at the measurement level. Binary RCA treats a product at RCA 1.01 as fully "in" and one at 0.99 as fully "out". If a report country's ECI rank hinged on one such product, a small data revision, a re-export correction or an HS-revision change could flip a sentence in the report. HS4 is the lab's planned upgrade (Q04, 11-14). Before the reports adopt it, we want to know which of their claims (level, rank among peers, direction of change) are robust to the threshold. The single-flip numbers also tell the optimizer discussion how much one product pushed across RCA = 1 is "worth" in ECI.

## Method (pre-stated before running)

**Baseline (H0):** oriented HS4 ECI per year: RCA ≥ 1, sorted SVD, O4 sign. ECI is z-scored across that year's countries, and rank 1 is the highest.

**Band (pre-stated):** b = 1.15. *Edge-in* products have RCA in [1, b), and *edge-out* products have RCA in [1/b, 1).

**Metrics (pre-stated):**
1. **Census:** edge-in and edge-out counts, edge-in share of the basket and of export value, and the mean PCI of each group vs the basket.
2. **Single flips (exact):** flip one band product of one country, recompute the full SVD for all countries, and record ΔECI (SD) and Δrank. We report the max |Δrank| and the max |ΔECI| per country. A fixed-PCI basket-mean prediction (11-22 §A) is computed alongside as an approximation check.
3. **Joint band (exact):** for one country at a time, set its threshold to b ("tight") or 1/b ("loose") and recompute everything. The interval is the min/max over {base, tight, loose}. Bands 1.05, 1.10, 1.15 and 1.25 are all reported.
4. **Report claims:** (a) the ordering between the report country and each peer is *robust* if the higher country's interval minimum exceeds the lower country's interval maximum; (b) the 2019 → 2022 direction is *robust* if the 2019 and 2022 intervals don't overlap.

**Pre-stated "fragile" flags:** a single flip moving ≥ 3 ranks, or a joint-band rank width ≥ 5. Benchmark: the median one-year |ΔECI| across countries.

**Scope:** 2019 single flips for all 133 countries (context). The 20 focus countries (7 report + 13 distinct peers) for 2020–2022. All variants tried are shown, and there was no model selection.

## Setup

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2020, 2021, 2022]
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']          # Spring 2026 report countries
PEERS = {'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
         'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'],
         'VNM': ['SGP', 'MYS', 'PHL']}                                 # from the reports (as 11-14 / 11-15)
FOCUS = sorted(set(REPORT) | {p for v in PEERS.values() for p in v})
BAND = 1.15                                                            # pre-stated band: RCA in [1/BAND, BAND)

def svd_o4(Mv):
    """Sorted SVD of a binary numpy M (countries x products), O4 orientation (PCI negative with ubiquity),
    as 11-19..11-22. Zero columns are ignored. Returns (z-scored ECI array, z-scored PCI array over kept cols, kept mask)."""
    kc, kp = Mv.sum(1), Mv.sum(0); keep = kp > 0
    Mk = Mv[:, keep]; kpk = kp[keep]
    B = Mk / np.sqrt(kc[:, None] * kpk[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kpk)
    s = -np.sign(np.corrcoef(q, kpk)[0, 1]); k, q = k * s, q * s
    return (k - k.mean()) / k.std(), (q - q.mean()) / q.std(), keep

def rank_of(e, i):
    """1 = highest ECI."""
    return int((e > e[i]).sum() + 1)

XH = {y: R.repo_hs(y, 4) for y in YEARS}
RC, MB, E0, P0 = {}, {}, {}, {}
for y in YEARS:
    Rm = eci.rca(XH[y]); M = (Rm >= 1).astype(float)
    M = M.loc[M.sum(axis=1) > 0, M.sum(axis=0) > 0]
    RC[y], MB[y] = Rm.loc[M.index, M.columns], M
    e, q, _ = svd_o4(M.to_numpy())
    E0[y], P0[y] = pd.Series(e, M.index), pd.Series(q, M.columns)
print('lib: eci.py research.py | numpy', np.__version__, '| pandas', pd.__version__)
print('HS4 countries x products:', {y: MB[y].shape for y in YEARS})
print('Focus set (%d):' % len(FOCUS), FOCUS)
print('Missing from the HS4 matrix:', {y: [c for c in FOCUS if c not in MB[y].index] for y in YEARS})

lib: eci.py research.py | numpy 2.4.6 | pandas 3.1.1
HS4 countries x products: {2019: (133, 1224), 2020: (130, 1223), 2021: (136, 1223), 2022: (119, 1231), 2023: (131, 1230)}
Focus set (20): ['ARE', 'AUS', 'CAN', 'CZE', 'DEU', 'FRA', 'GBR', 'HUN', 'ITA', 'JPN', 'KOR', 'KWT', 'MYS', 'PHL', 'POL', 'QAT', 'SAU', 'SGP', 'USA', 'VNM']
Missing from the HS4 matrix: {2019: [], 2020: [], 2021: [], 2022: ['SAU'], 2023: []}


### A. Baseline: oriented HS4 ECI ranks of the report countries

In [2]:
rows = []
for y in YEARS:
    e = E0[y]; d = MB[y].sum(axis=1)
    rows.append({'year': y, 'n': len(e), 'corr(ECI, diversity)': np.corrcoef(e, d)[0, 1],
                 **{c: int(e.rank(ascending=False)[c]) if c in e.index else np.nan for c in REPORT}})
A = pd.DataFrame(rows); print(A.round(3).to_string(index=False))

 year   n  corr(ECI, diversity)  CAN  DEU  POL  SGP  ARE  GBR  VNM
 2019 133                 0.670   45    6   31    5   56   14   58
 2020 130                 0.622   45    6   34    5   56   11   57
 2021 136                 0.641   43    6   33    4   56    8   53
 2022 119                 0.575   40    7   32    4   59    9   57
 2023 131                 0.569   43    7   34    4   55    8   52


**Takeaway (A):** H0 reproduces 11-22's Vietnam rank (58 of 133 in 2019). ECI correlates positively with diversity every year (0.569–0.670), so the orientation is correct. All 20 focus countries are present every year except Saudi Arabia in 2022.

### B. Census of near-threshold products (2019)

In [3]:
def census(y, c):
    r = RC[y].loc[c]; p = P0[y]
    inn = r[(r >= 1) & (r < BAND)]; out = r[(r >= 1 / BAND) & (r < 1)]
    x = XH[y].loc[c, RC[y].columns]
    basket = r[r >= 1]
    return {'year': y, 'country': c, 'diversity': len(basket), 'edge-in n': len(inn),
            'edge-in % of basket': 100 * len(inn) / len(basket),
            'edge-in % of export value': 100 * x[inn.index].sum() / x.sum(),
            'edge-out n': len(out), 'mean PCI basket': p[basket.index].mean(),
            'mean PCI edge-in': p[inn.index].mean() if len(inn) else np.nan,
            'mean PCI edge-out': p[out.index].mean() if len(out) else np.nan}
CEN = pd.DataFrame([census(2019, c) for c in FOCUS]).drop(columns='year')
CEN['role'] = ['report' if c in REPORT else 'peer' for c in CEN.country]
print(CEN.round(3).to_string(index=False))
ALLC = pd.DataFrame([census(2019, c) for c in MB[2019].index])
print('\nAll %d countries, 2019: median edge-in n = %.0f, median edge-in %% of basket = %.1f, median edge-out n = %.0f'
      % (len(ALLC), ALLC['edge-in n'].median(), ALLC['edge-in % of basket'].median(), ALLC['edge-out n'].median()))

country  diversity  edge-in n  edge-in % of basket  edge-in % of export value  edge-out n  mean PCI basket  mean PCI edge-in  mean PCI edge-out   role
    ARE         82         14               17.073                      0.622          20           -0.366            -0.320             -0.453 report
    AUS        112         12               10.714                      0.829          13           -0.707            -0.250              0.083   peer
    CAN        274         38               13.869                      4.944          45           -0.203             0.117              0.256 report
    CZE        343         49               14.286                      4.285          53            0.348             0.525              0.212   peer
    DEU        492         69               14.024                      7.939          65            0.519             0.247              0.140 report
    FRA        446         67               15.022                     11.794          57     

**Takeaway (B):** every report country has a sizeable band. Edge-in products are 10.4% (Poland) to 17.3% (UK) of the basket, and edge-out products number 19 (Singapore) to 70 (UK). By value, edge-in products are only 0.6% (UAE) to 7.9% (Germany) of exports. The typical country has 12 edge-in and 12 edge-out products (medians over 133). The band's mean PCI is not systematically higher than the basket's. For Canada, both the edge-in (0.117) and the edge-out (0.256) products are more complex than its basket (−0.203). For Germany both are less complex (0.247 and 0.140 vs 0.519).

### C. Exact single-product flips (2019, all countries)

In [4]:
def flips(y, c):
    """Exact: flip each band product of country c one at a time and recompute ECI for everyone.
    Also the fixed-PCI basket-mean prediction (only c's basket changes; PCI kept at H0)."""
    M = MB[y]; Mv = M.to_numpy().copy(); i = M.index.get_loc(c)
    e0 = E0[y].to_numpy(); r0 = rank_of(e0, i); p0 = P0[y].to_numpy()
    bm = (Mv @ p0) / Mv.sum(1)                                        # basket means, exact monotone map to ECI (11-22 A)
    r = RC[y].loc[c]; band = r[((r >= 1) & (r < BAND)) | ((r >= 1 / BAND) & (r < 1))]
    out = []
    for prod, rv in band.items():
        j = M.columns.get_loc(prod); old = Mv[i, j]
        Mv[i, j] = 1 - old
        if Mv[i].sum() == 0:
            Mv[i, j] = old; continue
        e1, _, _ = svd_o4(Mv)
        bm1 = bm.copy(); bm1[i] = (Mv[i] @ p0) / Mv[i].sum()
        Mv[i, j] = old
        out.append({'year': y, 'country': c, 'product': prod, 'side': 'in->out' if old == 1 else 'out->in',
                    'RCA': rv, 'PCI': p0[j], 'export $M': XH[y].loc[c, prod] / 1e6,
                    'dECI exact': e1[i] - e0[i], 'drank exact': rank_of(e1, i) - r0,
                    'drank fixed-PCI': int((bm1 > bm1[i]).sum() + 1) - r0})
    return out
t = time.time()
FL19 = pd.DataFrame([o for c in MB[2019].index for o in flips(2019, c)])
print('2019 exact single flips, all countries: %d flips in %.0f s' % (len(FL19), time.time() - t))
f = FL19
print('Spearman(drank exact, drank fixed-PCI) = %.3f; exact == fixed-PCI in %.1f%% of flips; |diff| <= 1 in %.1f%%'
      % (stats.spearmanr(f['drank exact'], f['drank fixed-PCI'])[0], 100 * (f['drank exact'] == f['drank fixed-PCI']).mean(),
         100 * ((f['drank exact'] - f['drank fixed-PCI']).abs() <= 1).mean()))
def summ(g):
    return pd.Series({'flips': len(g), 'max |drank|': g['drank exact'].abs().max(),
                      'median |drank|': g['drank exact'].abs().median(),
                      'flips moving >=3 ranks': int((g['drank exact'].abs() >= 3).sum()),
                      'max |dECI|': g['dECI exact'].abs().max(), 'sum |dECI|': g['dECI exact'].abs().sum()})
SF = FL19.groupby('country').apply(summ)
print('\nFocus countries, 2019 (rank of 133; positive drank = falls):')
print(SF.loc[FOCUS].assign(rank=[int(E0[2019].rank(ascending=False)[c]) for c in FOCUS]).round(3).to_string())
print('\nAll countries, 2019: median of max |drank| = %.1f; median of max |dECI| = %.4f'
      % (SF['max |drank|'].median(), SF['max |dECI|'].median()))
print('\nLargest single flips among report countries + peers (2019):')
TOP = FL19[FL19.country.isin(FOCUS)].assign(a=lambda d: d['drank exact'].abs()).sort_values(['a', 'export $M'], ascending=False).head(15)
print(TOP.drop(columns=['a', 'year']).round(3).to_string(index=False))

2019 exact single flips, all countries: 4706 flips in 141 s
Spearman(drank exact, drank fixed-PCI) = 0.952; exact == fixed-PCI in 97.0% of flips; |diff| <= 1 in 99.7%

Focus countries, 2019 (rank of 133; positive drank = falls):
         flips  max |drank|  median |drank|  flips moving >=3 ranks  max |dECI|  sum |dECI|  rank
country                                                                                          
ARE       34.0          2.0             0.0                     0.0       0.051       0.660    56
AUS       25.0          1.0             1.0                     0.0       0.029       0.368    83
CAN       83.0          1.0             1.0                     0.0       0.020       0.478    45
CZE      102.0          1.0             0.0                     0.0       0.013       0.486    11
DEU      134.0          0.0             0.0                     0.0       0.009       0.391     6
FRA      124.0          0.0             0.0                     0.0       0.011      

**Takeaway (C):** single flips barely move the report countries: max |Δrank| is 0–2 and max |ΔECI| is 0.009–0.051. Across all 4,706 flips, the fixed-PCI basket-mean shortcut gives exactly the same rank change in 97.0% of cases (within 1 rank in 99.7%; Spearman 0.952), so it is a good cheap screen. The biggest moves in the focus set belong to the UAE's small-basket peers: Kuwait's 3815 (catalysts) costs 7 ranks, and Qatar's only band product, 7404 (copper waste), costs 6. Vietnam's 8542 costs 1 rank.

### D. Joint band: one country's whole band moves at once
"Tight" means the country's own threshold is raised to b (its edge-in products leave), and "loose" means it is lowered to 1/b (edge-out products join). Every other country is unchanged. This gives an upper bound on what threshold noise in *one country's own* data can do.

In [5]:
def joint(y, c, b):
    """Country-only threshold scenarios for c (everyone else unchanged):
    'tight' = c's RCA threshold raised to b (products in [1, b) leave); 'loose' = lowered to 1/b (near-misses join)."""
    M = MB[y]; i = M.index.get_loc(c); r = RC[y].loc[c].to_numpy()
    res = {}
    for nm, row in [('tight', (r >= b).astype(float)), ('loose', (r >= 1 / b).astype(float))]:
        Mv = M.to_numpy().copy(); Mv[i] = row
        e1, _, _ = svd_o4(Mv); res[nm] = (e1[i], rank_of(e1, i))
    return res
BANDS = [1.05, 1.10, 1.15, 1.25]
rows = []
for y in [2019] + CHK:
    for c in FOCUS:
        if c not in MB[y].index: continue
        i = MB[y].index.get_loc(c); e0 = E0[y].to_numpy()
        for b in BANDS:
            j = joint(y, c, b); ev = [e0[i], j['tight'][0], j['loose'][0]]; rv = [rank_of(e0, i), j['tight'][1], j['loose'][1]]
            rows.append({'year': y, 'country': c, 'band': b, 'ECI': e0[i], 'rank': rv[0],
                         'ECI tight': ev[1], 'ECI loose': ev[2], 'rank tight': rv[1], 'rank loose': rv[2],
                         'ECI lo': min(ev), 'ECI hi': max(ev), 'best rank': min(rv), 'worst rank': max(rv)})
JT = pd.DataFrame(rows); JT['rank width'] = JT['worst rank'] - JT['best rank']; JT['ECI width'] = JT['ECI hi'] - JT['ECI lo']
print('2019, pre-stated band 1.15 (each country perturbed alone; interval = min/max over base, tight, loose):')
print(JT.query('year == 2019 and band == 1.15').drop(columns=['year', 'band', 'ECI lo', 'ECI hi']).round(3).to_string(index=False))
print('\nTightening the threshold RAISES ECI for %d of %d focus countries in 2019 (their marginal products sit below their basket-mean PCI).'
      % ((JT.query('year == 2019 and band == 1.15')['ECI tight'] > JT.query('year == 2019 and band == 1.15')['ECI']).sum(), len(FOCUS)))
print('\nInterval width by band, 2019 (focus countries): median / max')
print(JT.query('year == 2019').groupby('band')[['rank width', 'ECI width']].agg(['median', 'max']).round(3).to_string())
print('\nFocus countries with rank width >= 5 at band 1.15, by year:')
print(JT.query('band == 1.15 and `rank width` >= 5')[['year', 'country', 'rank', 'best rank', 'worst rank', 'rank width']].to_string(index=False))

def peer_order(y, b=BAND):
    J = JT.query('year == @y and band == @b').set_index('country'); out = []
    for c in REPORT:
        for p in PEERS[c]:
            if c not in J.index or p not in J.index: continue
            hi, lo = (c, p) if J.loc[c, 'ECI'] > J.loc[p, 'ECI'] else (p, c)
            out.append({'year': y, 'report': c, 'peer': p, 'higher at H0': hi, 'gap (SD)': J.loc[hi, 'ECI'] - J.loc[lo, 'ECI'],
                        'margin (SD)': J.loc[hi, 'ECI lo'] - J.loc[lo, 'ECI hi'],
                        'robust at band': bool(J.loc[hi, 'ECI lo'] > J.loc[lo, 'ECI hi'])})
    return out
PO = pd.DataFrame([o for y in [2019] + CHK for o in peer_order(y)])
print('\nPeer ordering (report country vs each peer) at the 1.15 band, 2019 (margin = worst case of higher minus best case of lower):')
print(PO.query('year == 2019').drop(columns='year').round(3).to_string(index=False))
print('\nRobust pairs by year:', {y: f"{int(g['robust at band'].sum())}/{len(g)}" for y, g in PO.groupby('year')})
print('Non-robust pairs:'); print(PO[~PO['robust at band']].round(3).to_string(index=False))

2019, pre-stated band 1.15 (each country perturbed alone; interval = min/max over base, tight, loose):
country    ECI  rank  ECI tight  ECI loose  rank tight  rank loose  best rank  worst rank  rank width  ECI width
    ARE  0.149    56      0.126      0.097          57          57         56          57           1      0.052
    AUS -0.503    83     -0.633     -0.335          92          78         78          92          14      0.298
    CAN  0.462    45      0.340      0.600          49          40         40          49           9      0.260
    CZE  1.515    11      1.455      1.492          12          12         11          12           1      0.060
    DEU  1.843     6      1.954      1.768           6           6          6           6           0      0.185
    FRA  1.139    22      1.138      1.122          22          23         22          23           1      0.017
    GBR  1.439    14      1.520      1.412          10          14         10          14           4     

**Takeaway (D):** at b = 1.15 the median focus-country rank interval is 1.5 ranks wide. Canada (40–49), Vietnam (58–63) and the UK (10–14) are the report countries with real width. Among peers, Kuwait (39–54) and Australia (78–92) have the widest intervals. The direction isn't one-sided: tightening raises ECI for 7 of 20 countries (e.g. Germany 1.843 → 1.954, UK 1.439 → 1.520), because their marginal products are less complex than their basket. Peer orderings hold in 20/21 pairs in 2019–2021 and 17/20 in 2022. The pairs that fail are the close ones: Singapore vs Korea (2019 gap 0.153 SD) and the Gulf pairs.

### E. Who is fragile? All 133 countries, 2019

In [6]:
D19 = ALLC.set_index('country').join(SF).join(JT.query('year == 2019 and band == 1.15').set_index('country')[['rank width']])
D19['ECI'] = E0[2019]; D19['rank'] = E0[2019].rank(ascending=False)
print('Spearman across all 133 countries, 2019:')
for a, b in [('max |drank|', 'diversity'), ('max |drank|', 'ECI'), ('edge-in % of basket', 'diversity'), ('max |dECI|', 'diversity')]:
    d = D19[[a, b]].dropna(); r, pv = stats.spearmanr(d[a], d[b]); print(f'  {a:22s} vs {b:10s}: rho = {r:+.3f} (p = {pv:.2g})')
# rank movement depends on local crowding: ECI gap to neighbours
es = E0[2019].sort_values(ascending=False); nbr = pd.Series(np.r_[np.nan, -np.diff(es.to_numpy())], es.index)
D19['gap to next above'] = nbr
d = D19.dropna(subset=['max |drank|']); r, pv = stats.spearmanr(d['max |drank|'], d['rank'].sub(67).abs()); print(f'  max |drank| vs |rank - 67| (distance from middle): rho = {r:+.3f} (p = {pv:.2g})')
pct = {c: 100 * (SF['max |drank|'] <= SF.loc[c, 'max |drank|']).mean() for c in REPORT}
print('Countries with no band products (no flips):', int(D19['max |drank|'].isna().sum()))
print('Countries with a single flip moving >= 3 ranks: %d of %d; >= 5 ranks: %d' % ((SF['max |drank|'] >= 3).sum(), len(SF), (SF['max |drank|'] >= 5).sum()))
print(SF[SF['max |drank|'] >= 3].join(D19[['diversity', 'rank']]).sort_values('max |drank|', ascending=False)[['max |drank|', 'max |dECI|', 'diversity', 'rank']].head(12).to_string())
print('Spearman(max |dECI|, diversity) restricted: see above. Median diversity of countries with max |drank| >= 3: %.0f vs others %.0f'
      % (D19.loc[D19['max |drank|'] >= 3, 'diversity'].median(), D19.loc[D19['max |drank|'] < 3, 'diversity'].median()))
print('Percentile of max |drank| among 133 countries:', {c: round(v, 1) for c, v in pct.items()})
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(D19['rank'], D19['max |drank|'], s=12, c='#9aa5b1')
for c in FOCUS:
    ax[0].scatter(D19.loc[c, 'rank'], D19.loc[c, 'max |drank|'], s=26, c='#c2410c' if c in REPORT else '#2563eb')
    ax[0].annotate(c, (D19.loc[c, 'rank'], D19.loc[c, 'max |drank|']), fontsize=7, xytext=(3, 2), textcoords='offset points')
ax[0].set_xlabel('HS4 ECI rank, 2019 (1 = highest)'); ax[0].set_ylabel('largest single-flip rank move')
ax[0].set_title('Single-product flips (RCA within 15% of 1)', fontsize=10)
J = JT.query('year == 2019 and band == 1.15').set_index('country').loc[FOCUS].sort_values('rank')
yy = np.arange(len(J))
ax[1].hlines(yy, J['best rank'] - 0.3, J['worst rank'] + 0.3, color='#2563eb', lw=3)
ax[1].scatter(J['rank'], yy, c='k', s=14, zorder=3)
ax[1].set_yticks(yy); ax[1].set_yticklabels([f'{c}*' if c in REPORT else c for c in J.index], fontsize=8)
ax[1].invert_yaxis(); ax[1].set_xlabel('HS4 ECI rank, 2019'); ax[1].set_title('Rank interval, own RCA threshold 1/1.15 to 1.15 (* = report)', fontsize=10)
plt.tight_layout(); plt.savefig(FIG / 'fragility_2019.png', dpi=110); plt.close()
print('saved figures/fragility_2019.png')

Spearman across all 133 countries, 2019:
  max |drank|            vs diversity : rho = -0.588 (p = 1.6e-13)
  max |drank|            vs ECI       : rho = -0.402 (p = 2e-06)
  edge-in % of basket    vs diversity : rho = +0.408 (p = 1.1e-06)
  max |dECI|             vs diversity : rho = -0.896 (p = 2.6e-47)
  max |drank| vs |rank - 67| (distance from middle): rho = -0.146 (p = 0.096)
Countries with no band products (no flips): 2
Countries with a single flip moving >= 3 ranks: 20 of 131; >= 5 ranks: 6
         max |drank|  max |dECI|  diversity   rank
country                                           
COG             15.0       0.204         27   97.0
AGO             10.0       0.157         15  101.0
KWT              7.0       0.146         21   39.0
QAT              6.0       0.112         10   94.0
ZMB              5.0       0.043         94   90.0
GAB              5.0       0.117         24  113.0
AZE              4.0       0.077         43  114.0
BFA              4.0       0.113     

saved figures/fragility_2019.png


![fragility](figures/fragility_2019.png)

**Takeaway (E):** fragility is a small-basket phenomenon. Max single-flip |ΔECI| has a Spearman correlation of −0.896 with diversity (max |Δrank|: −0.588). Countries with a ≥ 3-rank flip have a median diversity of 97, against 170 for the rest. Position in the dense middle of the ranking adds little (ρ = −0.146, p = 0.096). The report countries sit at the 24th–85th percentile of max |Δrank|, with the UAE the most exposed (2 ranks, 84.7th percentile).

### F. Other years and the reports' 2019 → 2022 direction calls

In [7]:
rows = []
for y in CHK:
    FL = pd.DataFrame([o for c in FOCUS if c in MB[y].index for o in flips(y, c)])
    s = FL.groupby('country').apply(summ)
    for c in REPORT:
        rows.append({'year': y, 'country': c, 'max |drank|': s.loc[c, 'max |drank|'], 'max |dECI|': s.loc[c, 'max |dECI|'],
                     'edge-in n': census(y, c)['edge-in n']})
FY = pd.DataFrame(rows)
print('Report countries, single-flip fragility by year:')
print(pd.concat([SF.loc[REPORT, ['max |drank|', 'max |dECI|']].assign(year=2019).reset_index(), FY]).pivot(index='country', columns='year', values='max |drank|').to_string())
# observed movement: median one-year |dECI| on countries present in both years
mv = [];
for y0, y1 in [(2019, 2020), (2020, 2021), (2021, 2022)]:
    cc = E0[y0].index.intersection(E0[y1].index); mv.append((E0[y1][cc] - E0[y0][cc]).abs().median())
print('\nMedian one-year |dECI| (countries in both years), 2019-20, 20-21, 21-22:', np.round(mv, 3))
rows = []
for c in REPORT:
    j19 = JT.query('year == 2019 and band == 1.15 and country == @c').iloc[0]
    j22 = JT.query('year == 2022 and band == 1.15 and country == @c').iloc[0]
    d = E0[2022][c] - E0[2019][c]
    rows.append({'country': c, 'dECI 2019->22': d, 'max single-flip |dECI| (2019)': SF.loc[c, 'max |dECI|'],
                 '2019 interval': f"[{j19['ECI lo']:.3f}, {j19['ECI hi']:.3f}]", '2022 interval': f"[{j22['ECI lo']:.3f}, {j22['ECI hi']:.3f}]",
                 'direction survives (intervals disjoint)': bool(j22['ECI lo'] > j19['ECI hi'] or j22['ECI hi'] < j19['ECI lo'])})
DIR = pd.DataFrame(rows); print(DIR.round(3).to_string(index=False))
print('Directions that survive the own-threshold band: %d of 7' % DIR['direction survives (intervals disjoint)'].sum())
print('Report countries whose max single-flip |dECI| (2019) is below the 2019-20 median one-year |dECI| (%.3f): %d of 7'
      % (mv[0], (SF.loc[REPORT, 'max |dECI|'] < mv[0]).sum()))

Report countries, single-flip fragility by year:
year     2019  2020  2021  2022
country                        
ARE       2.0   2.0   1.0   2.0
CAN       1.0   1.0   1.0   1.0
DEU       0.0   0.0   0.0   0.0
GBR       1.0   1.0   0.0   1.0
POL       1.0   1.0   1.0   0.0
SGP       0.0   1.0   0.0   0.0
VNM       1.0   1.0   0.0   0.0

Median one-year |dECI| (countries in both years), 2019-20, 20-21, 21-22: [0.084 0.08  0.134]
country  dECI 2019->22  max single-flip |dECI| (2019)   2019 interval   2022 interval  direction survives (intervals disjoint)
    CAN         -0.022                          0.020  [0.340, 0.600]  [0.415, 0.543]                                    False
    DEU         -0.125                          0.009  [1.768, 1.954]  [1.634, 1.819]                                    False
    POL         -0.207                          0.009  [0.782, 0.843]  [0.620, 0.672]                                     True
    SGP         -0.038                          0.035  [2.107

**Takeaway (F):** the single-flip result is stable: report countries' max |Δrank| is 0–2 in every year 2019–2022. The median one-year |ΔECI| is 0.084, 0.080 and 0.134 for 2019→20, 20→21 and 21→22, larger than any report country's largest single flip. Own-threshold intervals are wide enough to blur most direction calls: only Poland (−0.207) and Singapore (−0.038) have disjoint 2019 and 2022 intervals. The UAE's −0.236 doesn't survive because its 2022 interval reaches down to −0.087. Singapore surviving is a narrow-band statement only: under 11-16's noise models its −0.038 change was "unchanged".

### G. Contrast with 11-22: Vietnam re-based alone vs everyone re-based

In [8]:
X = XH[2019]; d = [c for c in X.columns if c[:2] in ('25', '26', '27')]
wsh = X[d].to_numpy().sum() / X.to_numpy().sum(); vsh = X.loc['VNM', d].sum() / X.loc['VNM'].sum()
f = (1 - wsh) / (1 - vsh)
print('World share of ch. 25-27 = %.4f, Vietnam share = %.4f, f = %.3f, 1/f = %.3f' % (wsh, vsh, f, 1 / f))
j = joint(2019, 'VNM', 1 / f)
print('Vietnam alone, threshold raised to 1/f (its H1 re-basing applied only to its own row): rank %d -> %d, dECI = %+.3f'
      % (rank_of(E0[2019].to_numpy(), MB[2019].index.get_loc('VNM')), j['tight'][1], j['tight'][0] - E0[2019]['VNM']))
print('Products leaving Vietnam basket:', int(((RC[2019].loc['VNM'] >= 1) & (RC[2019].loc['VNM'] < 1 / f)).sum()))
print('\nRun time: %.0f s' % (time.time() - T0))

World share of ch. 25-27 = 0.1276, Vietnam share = 0.0212, f = 0.891, 1/f = 1.122
Vietnam alone, threshold raised to 1/f (its H1 re-basing applied only to its own row): rank 58 -> 63, dECI = -0.067
Products leaving Vietnam basket: 20

Run time: 281 s


**Takeaway (G):** the H1 factor recomputed here matches 11-22 (f = 0.891, 1/f = 1.122, 20 products leave). Applied to Vietnam's row only, it costs **5 ranks (58 → 63, −0.067 SD)**. 11-22 found a 22-rank fall (58 → 80) when the chapters were dropped and *every* country was re-based. So most of that fall comes from what the drop does to other countries and to PCI, not from Vietnam's own marginal products. (The two numbers come from different decompositions, so treat this as an order-of-magnitude contrast, not an exact split.)

## Caveats

- **The band (1.15) is a choice.** Section D reports 1.05–1.25. Widths grow with b (median 1.0 → 1.0–1.5 ranks, max 3 → 20). Who is fragile at the other bands was not tabulated country by country tonight.
- **One country perturbed at a time.** Real data problems (D1, HS-revision mix, re-exports) hit many countries at once. 11-16 and 11-22 show that common shocks move ranks far more. Tonight bounds only *own-row* threshold noise.
- **Exact but binary.** All flips use binary RCA. A continuous RCA (Q07; 11-22 #2) would remove the threshold altogether.
- **Per-year samples.** The 2019 → 2022 comparison uses each year's sample (133 vs 119 countries), as 11-14's H4 did. Sample churn adds its own movement (Q03, 11-14).
- **Nominal, exporter-reported HS4, reporters ≥ US$1B.** Commodity prices move RCAs near 1 for resource exporters, which is part of why the Gulf peers are fragile.
- **2023 suspect (D1).** It appears only in Section A. D4 doesn't apply at HS4.

## Helpful to know

- **The fixed-PCI basket-mean shortcut predicts exact single-flip rank changes 97.0% of the time** (99.7% within one rank). For single-cell perturbations, you can screen thousands of candidates without an SVD and recompute exactly only the large ones.
- **Dropping near-threshold products can raise ECI.** That happens whenever a country's marginal products are less complex than its average basket (7 of 20 focus countries, including Germany and the UK). Any "fragility" statement must use a two-sided interval.
- **Small baskets are the fragile ones.** Kuwait (21 products) and Qatar (10) can move 6–7 ranks on one product. The UAE report's peer comparisons are therefore the least stable of the seven (UAE vs Kuwait and UAE vs Qatar fail in 2021/2022).
- **Canada's HS4 rank is the most threshold-sensitive report claim** (40–49 at b = 1.15). Its near-misses are more complex than its basket.
- One exact HS4 recompute (133 × 1,224) takes about 30 ms, so exhaustive single-cell experiments over one year take about 2 minutes.

## New research avenues (also in `proposals/2026-11-23.md`)

1. **Common-shock version of tonight's map.** Perturb *all* countries' RCAs by one shared factor per HS2 chapter (e.g. ±10% on a chapter's world share, mimicking a price swing) and compute the report countries' rank intervals. This separates price-driven threshold noise (shared) from own-data noise (tonight). One night, exact SVDs.
2. **Singapore vs Korea and the Gulf peer sentences.** Those are the only peer orderings that fail the band. Check them under ATLAS ECI 2019–2022 and under continuous RCA (log1p, capped). If they also flip there, recommend the reports call those pairs "comparable" rather than ranking them.
3. **Canada's near-miss cluster.** 45 edge-out products with mean PCI 0.256. List them by chapter and check whether they are re-export or US-integration products (auto parts, ch. 84/87). If they are, Canada's HS4 rank is set by a value-chain effect that a report should mention.
4. **Basket-mean screen as a `lib/` helper.** Add `flip_screen(M, pci, cells)` (fixed-PCI rank changes) and an exact `flip_exact()` to `lib/eci.py` (a `lib/` change, so Will should approve first). Tonight's 97.0% agreement justifies it.

## Data issues / data needed

None new. Everything used is in `data/raw/`. Known issues that touch this result: D1 (2023 suspect, used only in Section A) and nominal values (RCAs near 1 move with commodity prices). Saudi Arabia is missing from the 2022 HS4 matrix (the reason wasn't checked tonight), so 2022 has 20 peer pairs, not 21.